# Short-Term Power Forecasting — ML Project Process Flow

---

## VS Code Visualization Options

Before reading the diagram below, here are your options for rendering it in VS Code. The diagram is written in **Mermaid**, which is a plain-text diagram syntax — the same format used in GitHub READMEs.

| Option | What it is | How to use |
|---|---|---|
| **Markdown Preview Mermaid Support** | VS Code extension by bierner | Install the extension, open this `.md` file, press `Ctrl+Shift+V` to open the Markdown preview. The diagram renders inline. Best for keeping everything inside VS Code. |
| **Mermaid Editor (online)** | Browser-based live editor at mermaid.live | Paste the raw Mermaid code block into the editor. Instant rendering, easy export to PNG or SVG. Good for sharing. |
| **Draw.io Integration** | VS Code extension by Henning Dieterichs | Full drag-and-drop diagram editor inside VS Code. More manual but gives you pixel-level control over layout. Better for polished documentation diagrams than for code-defined diagrams. |

**Recommendation:** Start with **Markdown Preview Mermaid Support** — it is the lowest friction option and keeps the diagram version-controlled as plain text alongside your code.

---

## Process Flow Diagram

```mermaid
flowchart TD

    %% ── DATA SOURCES ──────────────────────────────────────────
    subgraph SOURCES["Data Sources"]
        A1["ENTSO-E Actual load and generation SE1 / SE2 / SE3 / SE4 hourly from 2015"]
        A2["ENTSO-E Historical day-ahead load forecasts archive of TSO forecasts used as evaluation benchmark"]
        A3["SMHI / ERA5 Historical weather observations Temp · Wind · Irradiance"]
        A4["Open-Meteo Historical Forecast API NWP model output archive from 2017"]
        A5["Open-Meteo Live Forecast API Current NWP model run"]
        A6["ENTSO-E Live day-ahead load forecast current TSO forecast published today for tomorrow"]
        A7["ENTSO-E Latest actual load real-time measured load per bidding area SE1 to SE4"]
    end

    %% ── RAW DATA ───────────────────────────────────────────────
    subgraph RAW["data/raw  ·  data/external"]
        B1["Raw power time series per bidding area"]
        B2["Raw weather observations"]
        B3["Raw historical NWP forecasts"]
    end

    A1 --> B1
    A3 --> B2
    A4 --> B3

    %% ── PREPROCESSING ──────────────────────────────────────────
    subgraph PREP["data/interim  —  Preprocessing"]
        C1["Resample to common hourly resolution"]
        C2["Handle missing values interpolation / forward fill"]
        C3["Timezone alignment all to UTC"]
        C4["Outlier detection and removal"]
    end

    B1 --> C1
    B2 --> C1
    B3 --> C1
    C1 --> C2 --> C3 --> C4

    %% ── FEATURE ENGINEERING ────────────────────────────────────
    subgraph FEAT["data/processed  —  Feature Engineering"]
        D1["Calendar features Hour · Weekday · Month · Holiday flag"]
        D2["Lag features Load t-1, t-24, t-168 which is 1h, 1day, 1week ago"]
        D3["Rolling statistics 24h and 168h rolling mean and std of load"]
        D4["Weather features Temp · Wind speed/dir · Irradiance · Wind at 100m"]
        D5["Per-area features one feature set per SE1 · SE2 · SE3 · SE4"]
    end

    C4 --> D1 & D2 & D3 & D4
    D1 & D2 & D3 & D4 --> D5

    %% ── TRAIN / VAL / TEST SPLIT ───────────────────────────────
    subgraph SPLIT["Train / Validation / Test Split"]
        E1["Training set 2018 to 2021 approximately 70 percent used to fit model weights"]
        E2["Validation set 2022 approximately 15 percent used for hyperparameter tuning and early stopping"]
        E3["Test set 2023 approximately 15 percent held out touched only once for final evaluation"]
    end

    D5 --> E1 & E2 & E3

    %% ── MODEL TRAINING ─────────────────────────────────────────
    subgraph MODELS["src/models  —  Model Training & Comparison"]
        F1["Baseline Naive persistence yt = y t-24 sets minimum performance bar"]
        F2["Linear Ridge Regression fast and interpretable good for linear seasonality"]
        F3["Gradient Boosting XGBoost / LightGBM handles non-linearity and feature interactions well"]
        F4["LSTM / GRU Recurrent neural network learns temporal dependencies needs more data and tuning"]
        F5["Temporal Fusion Transformer TFT state-of-the-art for multi-horizon time series"]
    end

    E1 --> F1 & F2 & F3 & F4 & F5
    E2 --> F3 & F4 & F5

    %% ── EVALUATION ─────────────────────────────────────────────
    subgraph EVAL["src/evaluation  —  Evaluation"]
        G0["TSO benchmark ENTSO-E historical day-ahead load forecast or 'load same as yesterday' used as reference to beat"]
        G1["Metrics per model: MAE Mean Absolute Error · RMSE penalises large errors · MAPE percent error scale-free · Pinball loss if probabilistic"]
        G2["Per-area breakdown SE1 · SE2 · SE3 · SE4 identify where model struggles geographically"]
        G3["Per-horizon breakdown 1h · 6h · 12h · 24h ahead identify degradation with forecast horizon"]
        G4["Select best model based on validation MAE / RMSE vs TSO benchmark"]
    end

    A2 --> G0
    B1 --> G0
    F1 & F2 & F3 & F4 & F5 --> G1
    G0 --> G1
    G1 --> G2 --> G3 --> G4

    %% ── FINAL TEST EVALUATION ──────────────────────────────────
    subgraph FINAL["Final Test Evaluation"]
        H1["Evaluate selected model on held-out test set 2023 report final MAE · RMSE · MAPE per area and horizon"]
    end

    G4 --> H1
    E3 --> H1

    %% ── MODEL SERIALISATION ────────────────────────────────────
    subgraph SERIALISE["models/trained"]
        I1["Serialise trained model as .pkl / .pt / .joblib save feature pipeline and scaler alongside model"]
    end

    H1 --> I1

    %% ── LIVE INFERENCE PIPELINE ────────────────────────────────
    subgraph LIVE["pipelines/  —  Live Inference"]
        J1["Fetch live NWP forecast from Open-Meteo Live API Temp · Wind · Irradiance"]
        J2["Fetch latest actual load from ENTSO-E API for lag features"]
        J6["Fetch live day-ahead TSO load forecast from ENTSO-E to compare against model output"]
        J3["Apply same feature engineering pipeline as training"]
        J4["Load serialised model run inference generate forecast for SE1 to SE4"]
        J5["Output forecast as CSV / dashboard / API with uncertainty bounds if probabilistic model"]
    end

    A5 --> J1
    A6 --> J6
    A7 --> J2
    I1 --> J4
    J1 --> J3
    J2 --> J3
    J3 --> J4 --> J5
    J6 --> J5

    %% ── MONITORING ─────────────────────────────────────────────
    subgraph MONITOR["Monitoring & Retraining"]
        K1["Compare forecast vs actual as new actuals arrive track MAE / RMSE over time"]
        K2["Detect performance drift retrain on expanded dataset when metrics degrade"]
    end

    J5 --> K1
    B1 --> K1
    K1 --> K2
    K2 --> E1
```
---

## Notes on Model Choices

**Naïve persistence baseline** should always be your first implementation. If your model cannot beat "the load will be the same as this time yesterday", something is wrong. It costs nothing to implement and sets the floor.

**Gradient Boosting (XGBoost / LightGBM)** is typically the strongest out-of-the-box model for tabular time series with engineered features. It handles the non-linear relationships between weather and load well, is robust to outliers, and trains fast. This is the recommended starting point for your first real model.

**LSTM / GRU** can capture sequential dependencies without manual lag feature engineering, but requires significantly more data, longer training times, and careful regularisation. Worth adding to the comparison once the XGBoost baseline is solid.

**Temporal Fusion Transformer (TFT)** is the current state-of-the-art for multi-horizon time series forecasting. It handles multiple forecast horizons simultaneously and produces interpretable attention weights. It is the most complex to implement — best treated as the final step of the comparison.

---

## Notes on the Train / Validation / Test Split

For time series data, the split must always be **chronological** — never random. Randomly shuffling time series data causes data leakage because future observations end up in the training set. The split above uses 2018–2021 for training, 2022 for validation, and 2023 as the held-out test set. The test set should be touched only once, at the very end, to report your final metrics honestly.

---

## Notes on Evaluation Metrics

**MAE (Mean Absolute Error)** is the most interpretable — it tells you the average error in MW, directly comparable to the scale of the load signal.

**RMSE (Root Mean Squared Error)** penalises large individual errors more heavily than MAE. In power systems, large forecast errors are operationally more costly than small ones, so RMSE is worth tracking alongside MAE.

**MAPE (Mean Absolute Percentage Error)** is scale-free, making it useful for comparing performance across SE1 (low load, northern Sweden) and SE4 (high load, southern Sweden) where absolute load levels differ significantly.
